In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Mar 31, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    build_feature_combos,
    compute_batch_size,
    detect_device,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_C'

df_train, df_val, df_test = load_split_bundle(CLEAN_DATA, DATA_SET)

OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  MAX_BATCH=65,536  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=65,536  adaptive BATCH_SIZE=32,768  INIT_LR=0.002828  n_train=1,716,781  steps/epoch~52
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

feature_combos = build_feature_combos(FEATURES_3, EXTRA_FEATURES, max_extra=3)

n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')

Total feature combinations: 130
  Base (3F):  1
  3F + 1:     9
  3F + 2:     36
  3F + 3:     84


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.59 GB / 85 GB  |  Free: 84.5 GB
Train: 1,716,781  Val: 490,509  Test: 245,255  Features: 12


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 64.9134  RMSE = 0.016269
Coefficients: a = -0.078535, b = -0.092385, c = -0.081882


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 130 MODELS
  GPU: A100-80GB  |  Batch: 32,768  |  AMP: True  |  Max epochs: 100

  [  1/130] 3F                             SSE=63.0227  Gain=+2.9%  ep=100  19.5s  elapsed=0.3min
  [ 25/130] 3F+iv_lag+rho                  SSE=41.1728  Gain=+36.6%  ep=100  13.0s  elapsed=5.4min
  [ 50/130] 3F+vix_lag+iv_lag+gamma        SSE=28.0727  Gain=+56.8%  ep=100  12.7s  elapsed=10.7min
  [ 75/130] 3F+iv_lag+d_iv_lag+vix_mom_lag SSE=32.7894  Gain=+49.5%  ep=100  12.4s  elapsed=16.0min
  [100/130] 3F+d_iv_lag+vix_mom_lag+rho    SSE=33.6191  Gain=+48.2%  ep=100  12.5s  elapsed=21.4min
  [125/130] 3F+vix_mom+theta+rho           SSE=52.5643  Gain=+19.0%  ep=100  13.0s  elapsed=26.6min
  [130/130] 3F+theta+vega+rho              SSE=54.0547  Gain=+16.7%  ep=100  12.6s  elapsed=27.7min

Done: 27.7 min for 130 models (avg 12.8s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,64.913429,0.000265,0.016269,0.005912,0.001246,0.002637,0.068266,None,None,None
1,3F,63.022717,0.000257,0.016030,0.007077,-0.000219,0.004079,0.095404,13.7s,2.91%,None
2,3F+vix_lag,72.143265,0.000294,0.017151,0.009679,0.002207,0.006459,-0.035508,13.2s,-11.14%,-14.47%
3,3F+iv_lag,59.076492,0.000241,0.015520,0.009743,0.001023,0.006774,0.152046,12.8s,8.99%,18.11%
4,3F+d_iv_lag,59.833759,0.000244,0.015619,0.009884,-0.000277,0.006713,0.141177,12.4s,7.83%,-1.28%
...,...,...,...,...,...,...,...,...,...,...,...
127,3F+gamma+theta+vega,51.411697,0.000210,0.014478,0.007841,0.002005,0.005187,0.262063,12.5s,20.80%,9.66%
128,3F+gamma+theta+rho,61.014832,0.000249,0.015773,0.009111,0.000482,0.006238,0.124224,12.5s,6.01%,-18.68%
129,3F+gamma+vega+rho,57.466217,0.000234,0.015307,0.008197,0.001938,0.005463,0.175159,12.6s,11.47%,5.82%
130,3F+theta+vega+rho,54.054653,0.000220,0.014846,0.007896,-0.000095,0.005224,0.224127,12.6s,16.73%,5.94%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,3F+iv_lag+gamma+vega,6,23.1480,0.009715,64.34,13.0
1,3F+iv_lag+gamma+theta,6,25.4862,0.010194,60.74,12.8
2,3F+vix_lag+iv_lag+d_iv_lag,6,27.2027,0.010532,58.09,12.5
3,3F+vix_lag+iv_lag+gamma,6,28.0727,0.010699,56.75,12.7
4,3F+vix_lag+iv_lag+theta,6,29.0274,0.010879,55.28,12.8
5,3F+vix_lag+iv_lag+vix_mom_lag,6,29.3154,0.010933,54.84,12.7
6,3F+vix_lag+iv_lag+vix_mom,6,30.0097,0.011062,53.77,12.9
7,3F+iv_lag+d_iv_lag+vix_mom,6,30.2493,0.011106,53.40,12.3
8,3F+vix_lag+iv_lag+rho,6,30.2772,0.011111,53.36,12.6
9,3F+d_iv_lag+vix_mom_lag+vix_mom,6,30.3700,0.011128,53.21,12.7


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): 3F
    SSE=63.0227  RMSE=0.016030  Gain=2.91%

+1 (4F): 3F+iv_lag
    SSE=59.0765  RMSE=0.015520  Gain=8.99%

+2 (5F): 3F+vix_lag+iv_lag
    SSE=30.8690  RMSE=0.011219  Gain=52.45%

+3 (6F): 3F+iv_lag+gamma+vega
    SSE=23.1480  RMSE=0.009715  Gain=64.34%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 27.6 min (0.46 hr)
Models trained: 130
Best overall: 3F+iv_lag+gamma+vega (Gain=64.34%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.81 GB / 85 GB
Total training time: 27.6 min for 130 models
